# UrbanSense AI: train air-quality models in Colab
Run the cells **in order**. Before you start, on your own computer: run the collectors so your data is fresh
(`python -m scripts.run_collector air-quality` and `weather` in `backend/`).
When asked to upload, select **both**: the latest `UrbanSense-AI-phase*.zip` (the code) and `urbansense_dev.db` (it sits in the project's root folder).

In [ ]:
from google.colab import files
import zipfile, shutil
up = files.upload()
zips = [f for f in up if f.endswith('.zip')]; dbs = [f for f in up if f.endswith('.db')]
assert zips and dbs, 'Select BOTH the project .zip and urbansense_dev.db'
with zipfile.ZipFile(zips[0]) as z: z.extractall('/content')
shutil.copy(dbs[0], '/content/UrbanSense-AI/urbansense_dev.db')
print('Project ready in /content/UrbanSense-AI')

In [ ]:
!pip install -q -r /content/UrbanSense-AI/requirements-ml.txt

## Train (a few minutes). Read the printed lines: they are the REAL held-out test results.

In [ ]:
%cd /content/UrbanSense-AI
!python -m ml_engine.pollution.train

## Forecast (needs data newer than 6 hours; if it says nothing was stored, refresh the data locally and re-upload the .db)

In [ ]:
%cd /content/UrbanSense-AI/backend
!python -m scripts.run_forecasts || echo "No forecasts stored: latest data is older than 6 h."
%cd /content/UrbanSense-AI
!python -m ml_engine.pollution.exchange export-predictions ml_engine/saved_models/pollution/predictions_$(date +%Y%m%d-%H%M).json

In [ ]:
print(open('/content/UrbanSense-AI/docs/ML_REPORT_air_quality.md').read())

## Download the results, extract them into your project root (overwrite), then run `python -m scripts.import_ml_outputs --all` in `backend/`.

In [ ]:
!cd /content/UrbanSense-AI && zip -qr /content/ml_outputs.zip ml_engine/saved_models/pollution docs/ML_REPORT_air_quality.md
files.download('/content/ml_outputs.zip')